# 04 - Parse dbt run_results.json to Delta Table
Ghi log dbt run vào bảng `silver_dbt_run_logs` trong Fabric Lakehouse.

In [ ]:
import json
import os
from datetime import datetime
from pyspark.sql import Row

dbt_project_dir = os.path.abspath("../dbt_project")
run_results_path = os.path.join(dbt_project_dir, "target", "run_results.json")

if not os.path.exists(run_results_path):
    print(f"⚠️ Không tìm thấy file {run_results_path}.")
    import sys
    sys.exit(0)

with open(run_results_path, "r", encoding="utf-8") as f:
    run_results = json.load(f)

metadata = run_results.get("metadata", {})
results = run_results.get("results", [])

invocation_id = metadata.get("invocation_id")
generated_at = metadata.get("generated_at")
args = run_results.get("args", {})
is_full_refresh = args.get("full_refresh", False)
load_type_val = "FULL" if is_full_refresh else "INCREMENTAL"

parsed_records = []
for res in results:
    unique_id = res.get("unique_id", "")
    status = res.get("status", "")
    message = res.get("message", "")
    execution_time_raw = res.get("execution_time")
    execution_time = float(execution_time_raw) if execution_time_raw is not None else 0.0
    adapter_response = res.get("adapter_response") or {}
    rows_affected_raw = adapter_response.get("rows_affected")
    rows_affected = int(rows_affected_raw) if rows_affected_raw is not None else 0
    
    parsed_records.append(Row(
        invocation_id=invocation_id,
        generated_at=generated_at,
        load_type=load_type_val,
        unique_id=unique_id,
        status=status,
        message=message,
        execution_time_seconds=execution_time,
        rows_affected=rows_affected,
        logged_at=datetime.now()
    ))

if parsed_records:
    df_logs = spark.createDataFrame(parsed_records)
    table_name = "workspace.silver.silver_dbt_run_logs"
    df_logs.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
    print(f"🎉 Đã lưu {len(parsed_records)} logs thành công vào bảng {table_name}")
else:
    print("Không có kết quả nào để lưu.")